# Introduction to Futures

This notebook provides an introduction to basic futures concepts
with examples from real market data.

By the end of this notebook, you should understand how to retrieve
data for the following:
* settlement (and why it is not the close)
* margin
* mark-to-market
* open interest

First, we will track a hypothetical trade through
daily settlement and final settlement to
understand the accounting of futures trading.
These are the building blocks of any historical
trade analysis.

Then we will take higher level view of different markets,
their size, contract specs, and expiration cycles.
You should begin to notice the similarities and
differences among different futures markets. In particular,
we will look at how volume and open interest
change over time and expiration in different markets.

There are also Python tips mixed in like managing
timestamps and timezones when working with market data.

In [1]:
# Import and check venv
import datetime
import sys
from collections.abc import Iterable
from dataclasses import dataclass, replace
from functools import reduce
from itertools import cycle
from typing import Any
from zoneinfo import ZoneInfo

import databento as db
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from finm37000 import (
    as_ct,
    disk_cache,
    get_databento_api_key,
    get_official_stats,
    make_ohlcv,
    temp_env,
    us_business_day,
)

pio.templates.default = "plotly_dark"
px.defaults.color_discrete_sequence = px.colors.qualitative.Set3  # type: ignore[attr-defined]
color_palette = cycle(px.defaults.color_discrete_sequence)  # type: ignore[attr-defined]

sys.executable

'/Users/ericpatterson/Documents/github/finm37000-main/.venv/bin/python'

In [2]:
# Construct databento client to retrieve data.

with temp_env(DATABENTO_API_KEY=get_databento_api_key()):
    client = db.Historical()

## Data For Tracking A Futures Trade

### Trades for the front Crude contract

Let's look at data for the April 2026 Crude futures contract on 2026-03-02.

In [3]:
cme = "GLBX.MDP3"
current_crude = "CLK6"
current_gold = "GCM6"
off_month_gold = "GCK6"
live_crude = "CLX6"
current_crude_and_gold_month = "May 2026"
es_symbols = (
    "ESH6",
    "ESM6",
    "ESU6",
    "ESZ6",
)
es_descriptor = "ES 2026"
soybean_symbols = (
    "ZSF6",
    "ZSH6",
    "ZSK6",
    "ZSN6",
    "ZSQ6",
    "ZSU6",
    "ZSX6",
    "ZSF7",
    "ZSH7",
)
soybean_descriptor = "2026 Soybean"
sofr_symbols = (
    "SR3H6",
    "SR3J6",
    "SR3K6",
    "SR3M6",
    "SR3N6",
)
euro_symbols = (
    "6EH6",
    "6EJ6",
    "6EK6",
    "6EM6",
    "6EN6",
)
quarterly_contracts = ("SR3H6", "SR3M6", "6EH6", "6EM6")
quarterly_and_serial_months = "March through July 2026"
serial_months = "April, May, and July 2026"
tz_chicago = ZoneInfo("America/Chicago")
now = datetime.datetime.now(tz=tz_chicago)
today = now.date()
trade_datetime = pd.Timestamp(2026, 4, 16, 11, 0, 0, tzinfo=tz_chicago)
trade_end = trade_datetime + pd.Timedelta(seconds=1)
trade_end_of_day = trade_datetime + pd.Timedelta(hours=12)
start_of_last_year = datetime.date(2025, 1, 1)
start_of_this_year = datetime.date(2026, 1, 1)
serial_date = datetime.date(
    2026, 3, 1
)  # short-lived serial instrument defs are not available at the start
yesterday = today - 2 * us_business_day

In [4]:
cache_scope = "futures_intro"


@disk_cache(scope=cache_scope)
def cached_trades(
    dataset: str,
    start: pd.Timestamp | datetime.datetime,
    end: pd.Timestamp | datetime.datetime,
    symbols: Iterable[str],
) -> pd.DataFrame:
    return client.timeseries.get_range(
        dataset=dataset,
        start=start,
        end=end,
        symbols=symbols,
        schema="trades",
    ).to_df()


trades = cached_trades(
    dataset=cme,
    start=trade_datetime,
    end=trade_end,
    symbols=current_crude,
)

In [5]:
some_trade_cols = [
    "side",
    "price",
    "size",
]
trades[some_trade_cols]

,side,price,size
ts_recv,,,
2026-04-16 16:00:00.151769594+00:00,A,93.36,1
2026-04-16 16:00:00.156910337+00:00,A,93.35,1


Consider the trade at \$93.35 at 11:00:00 (16:00:00 UTC).

The buyer and the seller must both have a margin account to finance this transaction.
Actual margin will depend on the trader's arrangements
with their broker or clearing firm.

As an example, let's assume that both buyer and seller
must have $10,000 in their margin accounts for this trade.

Why do we need \$10,000 to cover a price of \$93.35?
Futures prices reflect contract specifications including
a contract size. This data is found on exchange web sites and,
for automated workflows, in  symbol definition tables. For
example, here are some relevant columns from the  symbol
definition tables provided by the CME as passed through Databento.
I have limited to a small set of relevant columns for display; there
are many more.

In [6]:
client.metadata.list_fields(schema="definition", encoding="dbn")

/var/folders/8y/2f_yzjcd2tz12vswl92n4vpw0000gp/T/ipykernel_7562/2783403576.py:1: BentoWarning: No dataset was provided; the fields for the latest DBN encoding version (v3) are returned; some datasets may use a different version with different fields.
  client.metadata.list_fields(schema="definition", encoding="dbn")


[{'name': 'length', 'type': 'uint8_t'},
 {'name': 'rtype', 'type': 'uint8_t'},
 {'name': 'publisher_id', 'type': 'uint16_t'},
 {'name': 'instrument_id', 'type': 'uint32_t'},
 {'name': 'ts_event', 'type': 'uint64_t'},
 {'name': 'ts_recv', 'type': 'uint64_t'},
 {'name': 'min_price_increment', 'type': 'int64_t'},
 {'name': 'display_factor', 'type': 'int64_t'},
 {'name': 'expiration', 'type': 'uint64_t'},
 {'name': 'activation', 'type': 'uint64_t'},
 {'name': 'high_limit_price', 'type': 'int64_t'},
 {'name': 'low_limit_price', 'type': 'int64_t'},
 {'name': 'max_price_variation', 'type': 'int64_t'},
 {'name': 'unit_of_measure_qty', 'type': 'int64_t'},
 {'name': 'min_price_increment_amount', 'type': 'int64_t'},
 {'name': 'price_ratio', 'type': 'int64_t'},
 {'name': 'strike_price', 'type': 'int64_t'},
 {'name': 'raw_instrument_id', 'type': 'uint64_t'},
 {'name': 'leg_price', 'type': 'int64_t'},
 {'name': 'leg_delta', 'type': 'int64_t'},
 {'name': 'inst_attrib_value', 'type': 'int32_t'},
 {'na

In [7]:
@disk_cache(scope=cache_scope)
def cached_definitions(
    dataset: str,
    symbols: Iterable[str],
    date: datetime.date,
) -> pd.DataFrame:
    return client.timeseries.get_range(
        dataset=dataset,
        symbols=symbols,
        start=date,
        schema="definition",
    ).to_df()


current_crude_def = cached_definitions(
    dataset=cme,
    symbols=current_crude,
    date=trade_datetime.date(),
)

In [8]:
some_def_cols = [
    "instrument_id",
    "raw_symbol",
    "expiration",
    "unit_of_measure",
    "unit_of_measure_qty",
    "min_price_increment",
    "currency",
    "group",
    "exchange",
    "security_type",
    "high_limit_price",
    "low_limit_price",
]
current_crude_def[some_def_cols]

,instrument_id,raw_symbol,expiration,unit_of_measure,unit_of_measure_qty,min_price_increment,currency,group,exchange,security_type,high_limit_price,low_limit_price
ts_recv,,,,,,,,,,,,
2026-04-16 00:00:00+00:00,169926,CLK6,2026-04-21 18:30:00+00:00,BBL,1000.0,0.01,USD,CL,XNYM,FUT,NaN,NaN


For non-automated workflows, easier to understand on CME website
https://www.cmegroup.com/markets/energy/crude-oil/light-sweet-crude.contractSpecs.html

Key fields for size:
* `unit_of_measure_qty`: 1000,
* `unit_of_measure`: `BBL` for barrels,

\$93.35 is the unit price on 1000 barrels of crude oil,
i.e., one contract at this price represents \$933,500 of oil.

A long or short position only has to put up margin to start,
much less than full contract exposure.
Generally, exchange-set minimum margins are set to cover
the maximum possible one day loss if the market goes limit up or down
(see price limits: https://www.cmegroup.com/trading/price-limits.html),
but different market participants will face different margin requirements
depending on their financial arrangements.

### Which side benefits over the rest of the day
(assuming no additional trades by these counterparties)?

In [9]:
trades_until_midnight = cached_trades(
    dataset=cme,
    start=trade_datetime,
    end=trade_end_of_day,
    symbols=current_crude,
)

In [10]:
plot_cols = ("price", "size")
rule = "5min"
plot_names = (
    "Trade price",
    "Trade quantity",
)
fig = make_subplots(rows=len(plot_cols), cols=1, subplot_titles=list(plot_names))

for i, col in enumerate(plot_cols):
    fig.add_trace(
        go.Scatter(
            x=as_ct(trades_until_midnight.index),
            y=trades_until_midnight[col],
            mode="lines",
        ),
        row=i + 1,
        col=1,
    )

fig.update_layout(
    height=400,
    width=600,
    title_text=f"{current_crude} price activity post-trade",
    showlegend=False,
)
fig.show()

### Pre-quiz Yourself
The buyer and the seller will have their margin account
credited or debited at the end of the day.
Who do you think is getting the credit in this example
(i.e., who has a positive P&L on this trade for
the day)?

This is a pre-quiz because you cannot know the answer
with what has been provided in the notebook so far.
Look closely at the graph and take a guess.

[PollEv](https://www.polleverywhere.com/multiple_choice_polls/lzk7qOEhEuf0iCXCQeJzU)

In [11]:
plot_cols = ("price", "size")
agg_cols = ("volume",)
rule = "5min"
plot_names = ("Trade price", "Trade quantity", "Volume/5 minutes")
fig = make_subplots(
    rows=len(plot_cols) + len(agg_cols),
    cols=1,
    subplot_titles=list(plot_names),
)
agg_trades = make_ohlcv(trades_until_midnight, rule)

for i, col in enumerate(plot_cols):
    fig.add_trace(
        go.Scatter(
            x=as_ct(trades_until_midnight.index),
            y=trades_until_midnight[col],
            mode="lines",
        ),
        row=i + 1,
        col=1,
    )

for i, col in enumerate(agg_cols):
    fig.add_trace(
        go.Bar(x=as_ct(agg_trades.index), y=agg_trades[col]),
        row=len(plot_cols) + i + 1,
        col=1,
    )

fig.update_layout(
    height=600,
    width=600,
    title_text=f"{current_crude} price activity post-trade",
    showlegend=False,
)
fig.show()

The trade summary of activity shows the market closes at 16:00
and reopens an hour later with some after hours trading
but significantly less than during the trading day.
Do you notice anything else in the data?

### End-of-day Mark-to-market

Crude settles between 13:28 and 13:30 CT, then published around midnight.

#### Retrieving the official exchange-published settlement

Here are all the stats from the exchange during the remaining day.

In [12]:
@disk_cache(scope=cache_scope)
def cached_stats(
    dataset: str,
    symbols: Iterable[str],
    start: pd.Timestamp | datetime.date,
    end: pd.Timestamp | datetime.date | None = None,
) -> pd.DataFrame:
    return client.timeseries.get_range(
        dataset=dataset,
        schema="statistics",
        symbols=symbols,
        start=start,
        end=end,
    ).to_df()


raw_stats = cached_stats(
    dataset=cme,
    symbols=current_crude,
    start=trade_datetime,
    end=trade_end_of_day,
)
raw_stats

,ts_event,rtype,publisher_id,instrument_id,ts_ref,price,quantity,sequence,ts_in_delta,stat_type,channel_id,update_action,stat_flags,symbol
ts_recv,,,,,,,,,,,,,,
2026-04-16 16:33:44.111492822+00:00,2026-04-16 16:33:44.110732959+00:00,24,1,169926,NaT,93.99,9223372036854775807,478547311,17212,5,26,1,0,CLK6
2026-04-16 16:33:44.407796061+00:00,2026-04-16 16:33:44.407034469+00:00,24,1,169926,NaT,93.98,9223372036854775807,478551347,11809,8,26,1,0,CLK6
2026-04-16 16:33:44.813360879+00:00,2026-04-16 16:33:44.810167705+00:00,24,1,169926,NaT,93.99,9223372036854775807,478554999,11314,8,26,1,0,CLK6
2026-04-16 16:33:44.815076771+00:00,2026-04-16 16:33:44.813165749+00:00,24,1,169926,NaT,94.00,9223372036854775807,478555117,13319,5,26,1,0,CLK6
2026-04-16 16:33:44.822359967+00:00,2026-04-16 16:33:44.816050193+00:00,24,1,169926,NaT,94.03,9223372036854775807,478555413,16948,5,26,1,0,CLK6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-17 00:22:28.327907031+00:00,2026-04-17 00:22:28.327760025+00:00,24,1,169926,2026-04-16 00:00:00+00:00,NaN,90888,507750631,11031,9,26,1,0,CLK6
2026-04-17 00:22:28.327907031+00:00,2026-04-17 00:22:28.327760025+00:00,24,1,169926,2026-04-16 00:00:00+00:00,NaN,194354,507750631,11031,6,26,1,0,CLK6
2026-04-17 01:25:06.273236652+00:00,2026-04-17 01:25:06.273053145+00:00,24,1,169926,NaT,93.65,9223372036854775807,509850485,15709,8,26,1,0,CLK6


All the available statistic types:

In [13]:
[db.StatType.from_int(i) for i in range(1, 16)]

[<StatType.OPENING_PRICE: 1>,
 <StatType.INDICATIVE_OPENING_PRICE: 2>,
 <StatType.SETTLEMENT_PRICE: 3>,
 <StatType.TRADING_SESSION_LOW_PRICE: 4>,
 <StatType.TRADING_SESSION_HIGH_PRICE: 5>,
 <StatType.CLEARED_VOLUME: 6>,
 <StatType.LOWEST_OFFER: 7>,
 <StatType.HIGHEST_BID: 8>,
 <StatType.OPEN_INTEREST: 9>,
 <StatType.FIXING_PRICE: 10>,
 <StatType.CLOSE_PRICE: 11>,
 <StatType.NET_CHANGE: 12>,
 <StatType.VWAP: 13>,
 <StatType.VOLATILITY: 14>,
 <StatType.DELTA: 15>]

In [14]:
stat_cols = ["ts_ref", "price", "quantity", "stat_type", "stat_flags"]
raw_stats[stat_cols]

,ts_ref,price,quantity,stat_type,stat_flags
ts_recv,,,,,
2026-04-16 16:33:44.111492822+00:00,NaT,93.99,9223372036854775807,5,0
2026-04-16 16:33:44.407796061+00:00,NaT,93.98,9223372036854775807,8,0
2026-04-16 16:33:44.813360879+00:00,NaT,93.99,9223372036854775807,8,0
2026-04-16 16:33:44.815076771+00:00,NaT,94.00,9223372036854775807,5,0
2026-04-16 16:33:44.822359967+00:00,NaT,94.03,9223372036854775807,5,0
...,...,...,...,...,...
2026-04-17 00:22:28.327907031+00:00,2026-04-16 00:00:00+00:00,NaN,90888,9,0
2026-04-17 00:22:28.327907031+00:00,2026-04-16 00:00:00+00:00,NaN,194354,6,0
2026-04-17 01:25:06.273236652+00:00,NaT,93.65,9223372036854775807,8,0


There are seemingly many statistics in the exchange provided statistics,
but there are just a handful that
are an official reference value with a value in the `"ts_ref"` column

In [15]:
raw_stats[pd.notna(raw_stats["ts_ref"])][stat_cols]

,ts_ref,price,quantity,stat_type,stat_flags
ts_recv,,,,,
2026-04-16 18:30:52.033798942+00:00,2026-04-16 00:00:00+00:00,94.69,9223372036854775807,3,2
2026-04-16 21:38:17.750887273+00:00,2026-04-16 00:00:00+00:00,94.69,9223372036854775807,3,2
2026-04-16 23:16:09.054117825+00:00,2026-04-16 00:00:00+00:00,94.69,9223372036854775807,3,3
2026-04-17 00:22:28.327907031+00:00,2026-04-16 00:00:00+00:00,NaN,90888,9,0
2026-04-17 00:22:28.327907031+00:00,2026-04-16 00:00:00+00:00,NaN,194354,6,0


These are the settlement price (3), the open interest (9), and the cleared volume (6).

We will see in another notebook that
we can usually calculate the settlement price from the trade data,
but there can be exchange adjustments to
this value based on trades not in the electronic record.

The open interest requires knowledge of everyone's positions
and cannot be inferred from the trade data.

The cleared volume may include block trades and OTC transactions.


Note there are multiple settlements.

In [16]:
raw_stats[raw_stats["stat_type"] == db.StatType.SETTLEMENT_PRICE][stat_cols]

,ts_ref,price,quantity,stat_type,stat_flags
ts_recv,,,,,
2026-04-16 18:30:52.033798942+00:00,2026-04-16 00:00:00+00:00,94.69,9223372036854775807,3,2
2026-04-16 21:38:17.750887273+00:00,2026-04-16 00:00:00+00:00,94.69,9223372036854775807,3,2
2026-04-16 23:16:09.054117825+00:00,2026-04-16 00:00:00+00:00,94.69,9223372036854775807,3,3


The prices with `stat_flags==2` are preliminary settles.
Want to compare the official final settlement `stat_flags==3`.

For reference, see:
CME MDP3 tag 715 SettlPriceType flag: bit 0 = 1 (final) bit 1 = 1 (actual)
https://cmegroupclientsite.atlassian.net/wiki/spaces/EPICSANDBOX/pages/457414586/Settlement+Prices#SettlementPrices-SettlementatTradingTick/SettlementatClearingTick
https://cmegroupclientsite.atlassian.net/wiki/spaces/EPICSANDBOX/pages/457226917/MDP+3.0+-+Settlement+Price


Volumes and open interest are sometimes revised, and later values are more accurate.

The settle is so important that the exchange sends a flag when it is official

### Warning about dates without times

Real market data arrives with a date and a time,
so it is dangerous to use a date to specify
data unless you know it has already been preprocessed over the range you care about.

The raw data here is sensitive to when you get data.
In particular, if you just specify which
date you want, you do not necessarily get just data for that date,
and you may not get the official
numbers for the date you are looking for.

Databento makes a nice interface
that gives you flexibility about how you specify start and end time,
but I strongly advise using `datetime` or `pd.Timestamp`
with fully specified timezones to avoid
surprises, not just with Databento,
but with any work you do with intraday data.

N.B., `pd.Timestamp` supports nanosecond precision by default,
which matches most exchanges, whereas
`datetime` only goes to microseconds currently.

In [17]:
day_stats = cached_stats(
    dataset=cme,
    symbols=current_crude,
    start=trade_datetime.date(),
)

In [18]:
day_stats[pd.notna(day_stats["ts_ref"])][stat_cols]

,ts_ref,price,quantity,stat_type,stat_flags
ts_recv,,,,,
2026-04-16 00:22:23.227244380+00:00,2026-04-15 00:00:00+00:00,NaN,134031,9,0
2026-04-16 00:22:23.227244380+00:00,2026-04-15 00:00:00+00:00,NaN,240397,6,0
2026-04-16 15:06:15.045859388+00:00,2026-04-15 00:00:00+00:00,NaN,130892,9,0
2026-04-16 15:06:15.045859388+00:00,2026-04-15 00:00:00+00:00,NaN,240397,6,0
2026-04-16 18:30:52.033798942+00:00,2026-04-16 00:00:00+00:00,94.69,9223372036854775807,3,2
2026-04-16 21:38:17.750887273+00:00,2026-04-16 00:00:00+00:00,94.69,9223372036854775807,3,2
2026-04-16 23:16:09.054117825+00:00,2026-04-16 00:00:00+00:00,94.69,9223372036854775807,3,3


#### Continuing mark-to-market through expiration

The trade at \$93.35 at 11:00 settles at \$94.69 for the day.

The \$1.34 price difference per barrel is \$1,340 based on the contract size.

Futures buyer has \$1,340 credited to their margin account.
* Replenish (i.e., margin call) if the minimum level is breached.

Futures seller has \$1,340 debited from their margin account.

Let's calculate how that plays out assuming
our buyer and seller hold this position until final settlement.

Both buyer and seller can fund their margin accounts with a variety of assets:
https://www.cmegroup.com/solutions/clearing/financial-and-collateral-management/acceptable-collateral.html

Let's assume they both fund their margin accounts with T-bills,
receiving interest on their margin accounts with the clearing house getting,
e.g., a 0.5% cut.

When is final settlement, aka expiration?


In [19]:
current_crude_def["expiration"]

ts_recv
2026-04-16 00:00:00+00:00   2026-04-21 18:30:00+00:00
Name: expiration, dtype: datetime64[ns, UTC]

In [20]:
stats = get_official_stats(raw_stats, current_crude_def)
stats

,,Settlement price,Cleared volume,Open interest,expiration
Trade date,Symbol,,,,
2026-04-16,CLK6,94.69,194354.0,90888.0,2026-04-21 18:30:00+00:00


Here's another example of why you must
be careful with the timing of your queries. Requesting by date does not correspond
when the exchange disseminates information.

In [21]:
get_official_stats(day_stats, current_crude_def)

,,Settlement price,Cleared volume,Open interest,expiration
Trade date,Symbol,,,,
2026-04-15,CLK6,NaN,240397.0,130892.0,2026-04-21 18:30:00+00:00
2026-04-16,CLK6,94.69,NaN,NaN,2026-04-21 18:30:00+00:00


In [22]:
expiration = current_crude_def["expiration"].iloc[0]
raw_stats = cached_stats(
    dataset=cme,
    symbols=current_crude,
    start=trade_datetime.date(),
    end=expiration.date(),
)

In [23]:
stats = get_official_stats(raw_stats, current_crude_def)
stats

,,Settlement price,Cleared volume,Open interest,expiration
Trade date,Symbol,,,,
2026-04-15,CLK6,NaN,240397.0,130892.0,2026-04-21 18:30:00+00:00
2026-04-16,CLK6,94.69,194354.0,84571.0,2026-04-21 18:30:00+00:00
2026-04-17,CLK6,83.85,116624.0,59678.0,2026-04-21 18:30:00+00:00
2026-04-20,CLK6,89.61,NaN,NaN,2026-04-21 18:30:00+00:00


In [24]:
@dataclass
class Trade:
    position: int
    entry_datetime: pd.Timestamp
    entry_price: float
    initial_margin: float
    initial_margin_date: datetime.date
    maintenance_margin: float
    margin_interest_rate: float


def simulate_margin(
    settlements: pd.Series,
    trade: Trade,
    price_to_dollars: float,
    days_per_year: float = 365.0,
) -> pd.DataFrame:
    """Simulate daily margining for a single futures contract position.

    Args:
        settlements: Series containing settlements indexed by date and symbol.
        trade: Trade object with position, price, and margin information.
        price_to_dollars: Contract multiplier (e.g. 1000 for CL).
        days_per_year: Number of days per year to use when converting annual interest
                       rate to a daily rate.

    Returns:
        DataFrame with added columns for daily PnL, margin balance, and margin calls.

    """
    df = settlements.to_frame()
    df["last_value"] = settlements.shift(1)
    entry_date = trade.entry_datetime.date()
    df.loc[df.index.get_level_values(0) == entry_date, "last_value"] = trade.entry_price
    df["daily_pnl"] = (
        (settlements - df["last_value"]) * price_to_dollars * trade.position
    )
    df.iloc[0, df.columns.get_loc("daily_pnl")] = 0

    balances = np.ndarray(shape=(len(settlements),), dtype=float)
    margin_calls = balances.copy()
    interest_payments = balances.copy()
    last_balance = trade.initial_margin
    last_date = trade.initial_margin_date
    for i, (index, row) in enumerate(df.iterrows()):
        date = index[0]
        if date < last_date:
            continue
        days = (date - last_date).days
        last_date = date
        interest_payments[i] = (
            last_balance * trade.margin_interest_rate * days / days_per_year
        )
        last_balance += row["daily_pnl"]
        margin_calls[i] = 0
        if last_balance < trade.maintenance_margin:
            margin_calls[i] = trade.initial_margin - last_balance
            last_balance = trade.initial_margin
        balances[i] = last_balance
    df["margin_balance"] = balances
    df["margin_call"] = margin_calls
    df["margin_interest"] = interest_payments
    return df

The margins and P&L for the long side
assuming the margin account is funded with T-Bills paying 4% and
a 0.5% haircut on that and maintenance margin set at $9000.

In [25]:
t_bill_rate = 0.04
haircut = 0.005
long = Trade(
    position=1,
    entry_datetime=trade_datetime,
    entry_price=93.35,
    initial_margin=10_000,
    initial_margin_date=trade_datetime.date() - pd.Timedelta(days=1),
    maintenance_margin=9_000,
    margin_interest_rate=t_bill_rate - haircut,
)
long_margins = simulate_margin(
    stats["Settlement price"],
    long,
    price_to_dollars=1000,
)
long_margins

,,Settlement price,last_value,daily_pnl,margin_balance,margin_call,margin_interest
Trade date,Symbol,,,,,,
2026-04-15,CLK6,NaN,NaN,0.0,10000.0,0.0,0.000000
2026-04-16,CLK6,94.69,93.35,1340.0,11340.0,0.0,0.958904
2026-04-17,CLK6,83.85,94.69,-10840.0,10000.0,9500.0,1.087397
2026-04-20,CLK6,89.61,83.85,5760.0,15760.0,0.0,2.876712


The margins and P&L for the short side.

In [26]:
short = replace(long, position=-1)
short_margins = simulate_margin(
    stats["Settlement price"],
    short,
    price_to_dollars=1000,
)
short_margins

,,Settlement price,last_value,daily_pnl,margin_balance,margin_call,margin_interest
Trade date,Symbol,,,,,,
2026-04-15,CLK6,NaN,NaN,0.0,10000.0,0.0,0.000000
2026-04-16,CLK6,94.69,93.35,-1340.0,10000.0,1340.0,0.958904
2026-04-17,CLK6,83.85,94.69,10840.0,20840.0,0.0,0.958904
2026-04-20,CLK6,89.61,83.85,-5760.0,15080.0,0.0,5.995068


### Final Settlement

If both sides hold their position until expiration,
the buyer will buy physical crude at the settlement price,
and the seller will sell crude at the settlement price.

Actual delivery happens in Cushing, OK, with procedure defined by the exchange.
https://www.cmegroup.com/education/courses/introduction-to-crude-oil/crude-oil-fundamentals/delivery-of-wti-futures.html

The original buyer and seller do not necessarily complete the trade:
* either side may have exited their position
* the exchange will pair longs and shorts held until delivery
(not based on who traded).

This is nearly identical to a forward contract with the key difference being
the margin account and mark-to-market.

Note that not all final settlements lead to physical delivery like `CL`.
Many are financially settled.


## Overview of daily data for recent front month futures

Let's use the official statistics to see how much trading occurs in various markets.

### Recent Crude and Gold

Different markets have different lifetimes for their expirations.
* Crude: Monthly contracts for 10 years plus 2 months.
* Gold: Monthly contracts listed for 26 consecutive months
  and any Jun and Dec in the nearest 72 months.
* Euro FX: 20 quarters and 3 serials (i.e., non-quarterly months)

To get the whole history of the current crude contract, we need to go back 10 years.
Even though there is not a lot of data in each daily statistics
feed from the exchange, it takes several minutes
to retrieve the data, most of which is empty.
In the interest of speed, let's only go back
to the start of the year.

In [27]:
symbols = (
    current_crude,
    current_gold,
    off_month_gold,
)
raw_stats = cached_stats(
    dataset=cme,
    symbols=symbols,
    start=start_of_last_year,
    end=yesterday,
)

In [28]:
instrument_defs = cached_definitions(
    dataset=cme,
    symbols=symbols,
    date=trade_datetime.date(),
)
stats = get_official_stats(raw_stats, instrument_defs)

In [29]:
instrument_defs[some_def_cols]

,instrument_id,raw_symbol,expiration,unit_of_measure,unit_of_measure_qty,min_price_increment,currency,group,exchange,security_type,high_limit_price,low_limit_price
ts_recv,,,,,,,,,,,,
2026-04-16 00:00:00+00:00,19181,GCM6,2026-06-26 17:30:00+00:00,TRYOZ,100.0,0.10,USD,GC,XCEC,FUT,NaN,0.1
2026-04-16 00:00:00+00:00,42023114,GCK6,2026-05-27 17:30:00+00:00,TRYOZ,100.0,0.10,USD,GC,XCEC,FUT,NaN,0.1
2026-04-16 00:00:00+00:00,169926,CLK6,2026-04-21 18:30:00+00:00,BBL,1000.0,0.01,USD,CL,XNYM,FUT,NaN,NaN


(Hard to see, I know. Zooming will help.)

In [30]:
plot_df = stats.reset_index()
plot_groups = plot_df.groupby("Symbol")

x_col = "Trade date"
plot_cols = ["Cleared volume", "Open interest"]
symbols = list(plot_groups.groups.keys())
subplot_titles = [f"{symbol} {col}" for symbol in symbols for col in plot_cols]

fig = make_subplots(
    rows=len(symbols),
    cols=2,
    shared_xaxes="all",
    subplot_titles=subplot_titles,
)

for i, symbol in enumerate(symbols):
    df = plot_groups.get_group(symbol)
    fig.add_trace(
        go.Bar(
            x=df[x_col],
            y=df[plot_cols[0]],
        ),
        row=i + 1,
        col=1,
    )
    fig.add_trace(
        go.Bar(
            x=df[x_col],
            y=df[plot_cols[1]],
        ),
        row=i + 1,
        col=2,
    )

fig.update_layout(
    height=600,
    width=600,
    title_text=f"Cleared volume for {current_crude_and_gold_month} contracts",
    showlegend=False,
)
fig.show()

#### Actual vs. approximated trade value

What is the dollar value of all of this trading?

A simple approximation is to value the cleared volume at the settlement price. That is
the value of the contracts after everyone's margins are marked.

In [31]:
def dollarize_stats_at_settlement_price(
    stats: pd.DataFrame,
    instrument_defs: pd.DataFrame,
) -> pd.DataFrame:
    required_defs = ["unit_of_measure_qty"]
    if all(required in stats.columns for required in required_defs):
        extended_stats = stats.copy()
    else:
        extended_stats = stats.reset_index().merge(
            instrument_defs[["symbol"] + required_defs],
            left_on="Symbol",
            right_on="symbol",
        )
    extended_stats["Cleared volume($)"] = (
        extended_stats["unit_of_measure_qty"]
        * extended_stats["Settlement price"]
        * extended_stats["Cleared volume"]
    )
    extended_stats["Open interest($)"] = (
        extended_stats["unit_of_measure_qty"]
        * extended_stats["Settlement price"]
        * extended_stats["Open interest"]
    )
    cols = list(stats.columns)
    cols.extend(["unit_of_measure_qty", "Cleared volume($)", "Open interest($)"])
    extended_stats = extended_stats.set_index(["Trade date", "Symbol"])[cols]
    return extended_stats


approx_trade_value = dollarize_stats_at_settlement_price(stats, instrument_defs)
approx_trade_value

Settlement price  Cleared volume  Open interest  \
Trade date Symbol                                                    
2024-12-31 CLK6               66.95            98.0        10521.0   
           GCM6             2829.10             0.0          219.0   
2025-01-02 CLK6               67.58           313.0        10542.0   
           GCM6             2851.90             0.0          219.0   
2025-01-03 CLK6               67.89           288.0        10636.0   
...                             ...             ...            ...   
2026-06-22 GCM6             4181.90          1197.0          976.0   
2026-06-23 GCM6             4129.90           252.0          191.0   
2026-06-24 GCM6             3990.30           353.0          268.0   
2026-06-25 GCM6             4030.50          2724.0         2233.0   
2026-06-26 GCM6                 NaN             NaN            NaN   

                                 expiration  unit_of_measure_qty  \
Trade date Symbol                                                  
2024-12-31 CLK6   2026-04-21 18:30:00+00:00               1000.0   
           GCM6   2026-06-26 17:30:00+00:00                100.0   
2025-01-02 CLK6   2026-04-21 18:30:00+00:00               1000.0   
           GCM6   2026-06-26 17:30:00+00:00                100.0   
2025-01-03 CLK6   2026-04-21 18:30:00+00:00               1000.0   
...                                     ...                  ...   
2026-06-22 GCM6   2026-06-26 17:30:00+00:00                100.0   
2026-06-23 GCM6   2026-06-26 17:30:00+00:00                100.0   
2026-06-24 GCM6   2026-06-26 17:30:00+00:00                100.0   
2026-06-25 GCM6   2026-06-26 17:30:00+00:00                100.0   
2026-06-26 GCM6   2026-06-26 17:30:00+00:00                100.0   

                   Cleared volume($)  Open interest($)  
Trade date Symbol                                       
2024-12-31 CLK6         6.561100e+06       704380950.0  
           GCM6         0.000000e+00        61957290.0  
2025-01-02 CLK6         2.115254e+07       712428360.0  
           GCM6         0.000000e+00        62456610.0  
2025-01-03 CLK6         1.955232e+07       722078040.0  
...                              ...               ...  
2026-06-22 GCM6         5.005734e+08       408153440.0  
2026-06-23 GCM6         1.040735e+08        78881090.0  
2026-06-24 GCM6         1.408576e+08       106940040.0  
2026-06-25 GCM6         1.097908e+09       900010650.0  
2026-06-26 GCM6                  NaN               NaN  

[1045 rows x 7 columns]

In [32]:
plot_df = approx_trade_value.reset_index()
plot_groups = plot_df.groupby("Symbol")
x_col = "Trade date"
plot_cols = ["Cleared volume($)", "Open interest($)"]
symbols = list(plot_groups.groups.keys())
subplot_titles = [f"{symbol} {col}" for symbol in symbols for col in plot_cols]

fig = make_subplots(
    rows=len(symbols),
    cols=len(plot_cols),
    shared_xaxes="all",
    shared_yaxes=True,
    subplot_titles=subplot_titles,
)

for i, symbol in enumerate(symbols):
    df = plot_groups.get_group(symbol)
    for j, col in enumerate(plot_cols):
        fig.add_trace(
            go.Scatter(
                x=df[x_col],
                y=df[col],
            ),
            row=i + 1,
            col=1 + j,
        )


fig.update_layout(
    height=600,
    width=600,
    title_text="Dollar-value of cleared volume at settlement price",
    showlegend=False,
)
fig.show()

That does not reflect intraday variation in trade values. If you want that level of
detail, you may try to calculate it from the trade data, but first...


#### Cleared volume vs. observed trading volume

Before attempting a more accurate valuation of intraday trading,
let's validate the intraday trading volume against official volume.

I am switching to a live contract from the last week so that we can validate against
CME website.

In [33]:
a_week_ago = (today - 4 * us_business_day).date()
inclusive_end = yesterday + us_business_day
live_crude_raw_stats = cached_stats(
    dataset=cme,
    symbols=live_crude,
    start=a_week_ago,
    end=inclusive_end,
)
live_crude_def = cached_definitions(
    dataset=cme,
    symbols=live_crude,
    date=a_week_ago,
)
live_crude_stats = get_official_stats(live_crude_raw_stats, live_crude_def)

In [34]:
live_crude_stats

,,Settlement price,Cleared volume,Open interest,expiration
Trade date,Symbol,,,,
2026-09-23,CLX6,NaN,370714.0,311804.0,2026-10-20 18:30:00+00:00
2026-09-24,CLX6,94.61,410015.0,304677.0,2026-10-20 18:30:00+00:00
2026-09-25,CLX6,92.41,345783.0,294571.0,2026-10-20 18:30:00+00:00
2026-09-28,CLX6,92.60,NaN,NaN,2026-10-20 18:30:00+00:00


In [35]:
session_switch_chicago = datetime.time(16, 0)
trade_date = yesterday - us_business_day
session_start = datetime.datetime.combine(
    trade_date - datetime.timedelta(days=1),
    session_switch_chicago,
    tzinfo=tz_chicago,
)
session_end = datetime.datetime.combine(
    trade_date,
    session_switch_chicago,
    tzinfo=tz_chicago,
)
live_crude_raw_trades = cached_trades(
    dataset=cme,
    symbols=live_crude,
    start=session_start,
    end=session_end,
)

In [36]:
trade_date, live_crude_raw_trades["size"].sum()

(Timestamp('2026-09-25 00:00:00'), np.uint64(246225))

In [37]:
live_crude_stats["Cleared volume"]

Trade date  Symbol
2026-09-23  CLX6      370714.0
2026-09-24  CLX6      410015.0
2026-09-25  CLX6      345783.0
2026-09-28  CLX6           NaN
Name: Cleared volume, dtype: float64

We seem to be missing about 100,000 futures trades?!

Compare to the CME:
https://www.cmegroup.com/markets/energy/crude-oil/light-sweet-crude.volume.html

#### Settlements vs. Closes

Many sources confuse settlements and closes,
but the close is not used to mark-to-market,
and the time of the close is sensitive to what
you or your source call the end of the day.

In [38]:
@disk_cache(scope=cache_scope)
def cached_daily_ohlcv(
    dataset: str,
    symbols: Iterable[str],
    start: pd.Timestamp | datetime.date,
    end: pd.Timestamp,
) -> pd.DataFrame:
    return client.timeseries.get_range(
        dataset=dataset,
        schema="ohlcv-1d",
        symbols=symbols,
        start=start,
        end=end,
    ).to_df()


ohlcv = cached_daily_ohlcv(
    dataset=cme,
    symbols=symbols,
    start=start_of_last_year,
    end=yesterday,
)

In [39]:
plot_ohlcv = ohlcv.reset_index()
plot_ohlcv["Trade date"] = plot_ohlcv["ts_event"].dt.date
plot_ohlcv = plot_ohlcv.rename(columns={"symbol": "Symbol"})
plot_stats = stats.reset_index()
plot_df = plot_ohlcv.merge(plot_stats, on=["Symbol", "Trade date"])
plot_groups = plot_df.groupby("Symbol")
x_col = "Trade date"
subplot_titles = list(plot_groups.groups.keys())

fig = make_subplots(
    rows=len(subplot_titles),
    cols=1,
    shared_xaxes="all",
    subplot_titles=subplot_titles,
)
for i, symbol in enumerate(symbols):
    df = plot_groups.get_group(symbol)
    custom_data = df[["Settlement price", "close"]]
    hover_template = (
        "Close: %{customdata[0]}<br>"
        "Settle: %{customdata[1]}"
        "<extra></extra>"  # <extra></extra> removes trace name
    )
    fig.add_trace(
        go.Scatter(
            x=df[x_col],
            y=df["Settlement price"],
            customdata=custom_data,
            hovertemplate=hover_template,
            opacity=0.5,
        ),
        row=i + 1,
        col=1,
    )
    fig.add_trace(
        go.Candlestick(
            x=df[x_col],
            open=df["open"],
            high=df["high"],
            low=df["low"],
            close=df["close"],
        ),
        row=i + 1,
        col=1,
    )
    fig.update_xaxes(rangeslider_visible=False)

fig.update_layout(
    height=600,
    width=600,
    title_text=f"Close and Settlement for {', '.join(symbols)}",
    showlegend=False,
)
fig.show()

### Recent 3-month SOFR and Euro FX

These have monthly contracts, but quarterly contracts dominate trading.

In [40]:
futures_groups = {
    "SOFR": sofr_symbols,
    "Euro FX": euro_symbols,
}
symbols = reduce(lambda x, y: x + y, futures_groups.values())
raw_stats = cached_stats(
    dataset=cme,
    symbols=symbols,
    start=start_of_this_year,
    end=yesterday,
)

In [41]:
instrument_defs = cached_definitions(
    dataset=cme,
    symbols=symbols,
    date=serial_date,
)

In [42]:
stats = get_official_stats(raw_stats, instrument_defs)
instrument_defs[some_def_cols]

,instrument_id,raw_symbol,expiration,unit_of_measure,unit_of_measure_qty,min_price_increment,currency,group,exchange,security_type,high_limit_price,low_limit_price
ts_recv,,,,,,,,,,,,
2026-03-01 13:03:03.201627549+00:00,254277,SR3M6,2026-09-15 21:00:00+00:00,USD,2500.0,0.00500,USD,SS,XCME,FUT,NaN,0.00250
2026-03-01 13:03:03.209807051+00:00,42004020,SR3K6,2026-08-18 21:00:00+00:00,USD,2500.0,0.00500,USD,SS,XCME,FUT,NaN,0.00250
2026-03-01 13:03:03.221442776+00:00,42000061,SR3J6,2026-07-14 21:00:00+00:00,USD,2500.0,0.00500,USD,SS,XCME,FUT,NaN,0.00250
2026-03-01 13:03:03.229597454+00:00,254276,SR3H6,2026-06-16 21:00:00+00:00,USD,2500.0,0.00250,USD,SS,XCME,FUT,NaN,0.00250
2026-03-01 13:03:03.256604164+00:00,42056596,SR3N6,2026-10-20 21:00:00+00:00,USD,2500.0,0.00500,USD,SS,XCME,FUT,NaN,0.00250
2026-03-01 13:08:36.097903325+00:00,57062,6EH6,2026-03-16 14:16:00+00:00,EUR,125000.0,0.00005,USD,6E,XCME,FUT,NaN,0.00005
2026-03-01 13:08:36.102116007+00:00,42012835,6EK6,2026-05-18 14:16:00+00:00,EUR,125000.0,0.00005,USD,6E,XCME,FUT,NaN,0.00005
2026-03-01 13:08:36.206662739+00:00,20048,6EM6,2026-06-15 14:16:00+00:00,EUR,125000.0,0.00005,USD,6E,XCME,FUT,NaN,0.00005
2026-03-01 13:08:36.262557533+00:00,42004841,6EN6,2026-07-13 14:16:00+00:00,EUR,125000.0,0.00005,USD,6E,XCME,FUT,NaN,0.00005


In [43]:
plot_df = stats.reset_index()
plot_groups = plot_df.groupby("Symbol")
colors = {group: next(color_palette) for group in plot_groups.groups}

x_col = "Trade date"
plot_cols = ["Cleared volume", "Open interest"]
subplot_titles = [f"{family} {col}" for family in futures_groups for col in plot_cols]

fig = make_subplots(
    rows=len(futures_groups),
    cols=len(plot_cols),
    shared_xaxes="all",
    subplot_titles=subplot_titles,
)

for i, (_, group_symbols) in enumerate(futures_groups.items()):
    for symbol in group_symbols:
        df = plot_groups.get_group(symbol)
        fig.add_trace(
            go.Scatter(
                x=df[x_col],
                y=df[plot_cols[0]],
                name=symbol,
                line=dict(color=colors[symbol]),
            ),
            row=i + 1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=df[x_col],
                y=df[plot_cols[1]],
                name=symbol,
                line=dict(color=colors[symbol]),
                showlegend=False,
            ),
            row=i + 1,
            col=2,
        )

fig.update_layout(
    height=600,
    width=600,
    title_text=f"Cleared volume for {quarterly_and_serial_months} contracts",
)
fig.show()

In [44]:
plot_df = stats.reset_index()
plot_df = plot_df[~plot_df["Symbol"].isin(quarterly_contracts)]
plot_groups = plot_df.groupby("Symbol")
colors = {group: next(color_palette) for group in plot_groups.groups}

x_col = "Trade date"
plot_cols = ["Cleared volume", "Open interest"]
subplot_titles = [f"{family} {col}" for family in futures_groups for col in plot_cols]

fig = make_subplots(
    rows=len(futures_groups),
    cols=len(plot_cols),
    shared_xaxes="all",
    subplot_titles=subplot_titles,
)

for i, (_, group_symbols) in enumerate(futures_groups.items()):
    for symbol in group_symbols:
        if symbol not in plot_groups.groups:
            continue
        df = plot_groups.get_group(symbol)
        fig.add_trace(
            go.Scatter(
                x=df[x_col],
                y=df[plot_cols[0]],
                name=symbol,
                line=dict(color=colors[symbol]),
            ),
            row=i + 1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=df[x_col],
                y=df[plot_cols[1]],
                name=symbol,
                line=dict(color=colors[symbol]),
                showlegend=False,
            ),
            row=i + 1,
            col=2,
        )

fig.update_layout(
    height=600,
    width=600,
    title_text=f"Serial (non-quarterly) {serial_months} contracts",
)
fig.show()

### S&P 500 Mini Futures

These are quarterly for last 21 quarters.

In [45]:
symbols = es_symbols
raw_stats = cached_stats(
    dataset=cme,
    symbols=symbols,
    start=start_of_this_year,
    end=yesterday,
)

In [46]:
instrument_defs = cached_definitions(
    dataset=cme,
    symbols=symbols,
    date=start_of_this_year,
)

In [47]:
stats = get_official_stats(raw_stats, instrument_defs)
instrument_defs[some_def_cols]

,instrument_id,raw_symbol,expiration,unit_of_measure,unit_of_measure_qty,min_price_increment,currency,group,exchange,security_type,high_limit_price,low_limit_price
ts_recv,,,,,,,,,,,,
2026-01-01 00:00:00+00:00,42140870,ESU6,2026-09-18 13:30:00+00:00,IPNT,50.0,0.25,USD,ES,XCME,FUT,7565.50,6595.50
2026-01-01 00:00:00+00:00,10252,ESZ6,2026-12-18 14:30:00+00:00,IPNT,50.0,0.25,USD,ES,XCME,FUT,7613.50,6643.50
2026-01-01 00:00:00+00:00,42140878,ESH6,2026-03-20 13:30:00+00:00,IPNT,50.0,0.25,USD,ES,XCME,FUT,7464.25,6494.25
2026-01-01 00:00:00+00:00,42140864,ESM6,2026-06-18 13:30:00+00:00,IPNT,50.0,0.25,USD,ES,XCME,FUT,7516.00,6546.00


In [48]:
def make_volume_oi_title(descriptor: str) -> dict[str, Any]:
    return {
        "title_text": (
            f"Cleared volume and open interest<br>over time for {descriptor} contracts"
        ),
        "title_y": 0.95,  # add some space around multiline title
    }

In [49]:
plot_df = stats.reset_index()
plot_df["Symbol"].unique()
plot_df = stats.reset_index()
plot_groups = plot_df.groupby("Symbol")
colors = {group: next(color_palette) for group in plot_groups.groups}

x_col = "Trade date"
plot_cols = ["Cleared volume", "Open interest"]
subplot_titles = plot_cols

fig = make_subplots(
    rows=len(plot_cols),
    cols=1,
    shared_xaxes="all",
    subplot_titles=subplot_titles,
)

for symbol in symbols:
    df = plot_groups.get_group(symbol)
    fig.add_trace(
        go.Scatter(
            x=df[x_col],
            y=df[plot_cols[0]],
            name=symbol,
            line=dict(color=colors[symbol]),
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=df[x_col],
            y=df[plot_cols[1]],
            name=symbol,
            line=dict(color=colors[symbol]),
            showlegend=False,
        ),
        row=2,
        col=1,
    )

fig.update_layout(
    height=600,
    width=600,
    **make_volume_oi_title(es_descriptor),
)
fig.show()

### Soybeans

Agriculture futures often have unusual calendar cycles.


In [50]:
symbols = soybean_symbols
raw_stats = cached_stats(
    dataset=cme,
    symbols=symbols,
    start=start_of_this_year,
    end=yesterday,
)

In [51]:
instrument_defs = cached_definitions(
    dataset=cme,
    symbols=symbols,
    date=start_of_this_year,
)

In [52]:
stats = get_official_stats(raw_stats, instrument_defs)
instrument_defs[some_def_cols]

,instrument_id,raw_symbol,expiration,unit_of_measure,unit_of_measure_qty,min_price_increment,currency,group,exchange,security_type,high_limit_price,low_limit_price
ts_recv,,,,,,,,,,,,
2026-01-01 00:00:00+00:00,42011290,ZSQ6,2026-08-14 17:01:00+00:00,BU,5000.0,0.25,USD,ZS,XCBT,FUT,1162.50,1022.50
2026-01-01 00:00:00+00:00,42001323,ZSF6,2026-01-14 18:01:00+00:00,BU,5000.0,0.25,USD,ZS,XCBT,FUT,1128.75,988.75
2026-01-01 00:00:00+00:00,387199,ZSX6,2026-11-13 18:01:00+00:00,BU,5000.0,0.25,USD,ZS,XCBT,FUT,1152.25,1012.25
2026-01-01 00:00:00+00:00,42000277,ZSH7,2027-03-12 18:01:00+00:00,BU,5000.0,0.25,USD,ZS,XCBT,FUT,1161.75,1021.75
2026-01-01 00:00:00+00:00,194904,ZSN6,2026-07-14 17:01:00+00:00,BU,5000.0,0.25,USD,ZS,XCBT,FUT,1165.50,1025.50
2026-01-01 00:00:00+00:00,42009683,ZSK6,2026-05-14 17:01:00+00:00,BU,5000.0,0.25,USD,ZS,XCBT,FUT,1154.25,1014.25
2026-01-01 00:00:00+00:00,42011067,ZSH6,2026-03-13 17:01:00+00:00,BU,5000.0,0.25,USD,ZS,XCBT,FUT,1142.50,1002.50
2026-01-01 00:00:00+00:00,42000229,ZSF7,2027-01-14 18:01:00+00:00,BU,5000.0,0.25,USD,ZS,XCBT,FUT,1162.25,1022.25
2026-01-01 00:00:00+00:00,42011482,ZSU6,2026-09-14 17:01:00+00:00,BU,5000.0,0.25,USD,ZS,XCBT,FUT,1147.75,1007.75


In [53]:
plot_df = stats.reset_index()
plot_df["Symbol"].unique()
plot_df = stats.reset_index()
plot_groups = plot_df.groupby("Symbol")
colors = {group: next(color_palette) for group in plot_groups.groups}

x_col = "Trade date"
plot_cols = ["Cleared volume", "Open interest"]
subplot_titles = plot_cols

fig = make_subplots(
    rows=len(plot_cols),
    cols=1,
    shared_xaxes="all",
    subplot_titles=subplot_titles,
)

for symbol in symbols:
    df = plot_groups.get_group(symbol)
    fig.add_trace(
        go.Scatter(
            x=df[x_col],
            y=df[plot_cols[0]],
            name=symbol,
            line=dict(color=colors[symbol]),
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=df[x_col],
            y=df[plot_cols[1]],
            name=symbol,
            line=dict(color=colors[symbol]),
            showlegend=False,
        ),
        row=2,
        col=1,
    )

fig.update_layout(height=600, width=600, **make_volume_oi_title(soybean_descriptor))
fig.show()

## Next steps

You should now understand how to track and account for a futures trade in market data,
and you have seen how different expiration cycles and patterns from some major futures
markets.

These market examples have been chosen to represent some different market types:
energies (Crude), metals(Gold), interest rates (SOFR), foreign exchange (Euro FX),
equity indexes (ES), and agricultural products (Soybeans). They show the major themes
of different expiration cycles so that you will know what to look for when you
look at them or other futures markets. I recommend repeating these exercises
for other markets that interest you. For some of the biggest markets, consider

* Energies: Natural Gas (NG)
* Metals: Silver (SI) or Copper (HG)
* Interest rate: Ten Years (ZN)
* Foreign exchange: Yen (6J)
* Equity indexes: Nasdaq (NQ)
* Agricultural products: Corn (ZC)

Use the CME web site to get an idea which markets are most active: https://www.cmegroup.com/markets.html